# Subject: crashers
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/cpython/Lib/test/crashers

### File: bogus_code_obj.py

In [ ]:
"""
Broken bytecode objects can easily crash the interpreter.

This is not going to be fixed.  It is generally agreed that there is no
point in writing a bytecode verifier and putting it in CPython just for
this.  Moreover, a verifier is bound to accept only a subset of all safe
bytecodes, so it could lead to unnecessary breakage.

For security purposes, "restricted" interpreters are not going to let
the user build or load random bytecodes anyway.  Otherwise, this is a
"won't fix" case.

"""

def f():
    pass

f.__code__ = f.__code__.replace(co_code=b"")
f()

### File: gc_inspection.py

In [ ]:
"""
gc.get_referrers() can be used to see objects before they are fully built.

Note that this is only an example.  There are many ways to crash Python
by using gc.get_referrers(), as well as many extension modules (even
when they are using perfectly documented patterns to build objects).

Identifying and removing all places that expose to the GC a
partially-built object is a long-term project.  A patch was proposed on
SF specifically for this example but I consider fixing just this single
example a bit pointless (#1517042).

A fix would include a whole-scale code review, possibly with an API
change to decouple object creation and GC registration, and according
fixes to the documentation for extension module writers.  It's unlikely
to happen, though.  So this is currently classified as
"gc.get_referrers() is dangerous, use only for debugging".
"""

import gc


def g():
    marker = object()
    yield marker
    # now the marker is in the tuple being constructed
    [tup] = [x for x in gc.get_referrers(marker) if type(x) is tuple]
    print(tup)
    print(tup[1])


tuple(g())

### File: infinite_loop_re.py

In [ ]:
# This was taken from https://bugs.python.org/issue1541697
# It's not technically a crasher.  It may not even truly be infinite,
# however, I haven't waited a long time to see the result.  It takes
# 100% of CPU while running this and should be fixed.

import re
starttag = re.compile(r'<[a-zA-Z][-_.:a-zA-Z0-9]*\s*('
        r'\s*([a-zA-Z_][-:.a-zA-Z_0-9]*)(\s*=\s*'
        r'(\'[^\']*\'|"[^"]*"|[-a-zA-Z0-9./,:;+*%?!&$\(\)_#=~@]'
        r'[][\-a-zA-Z0-9./,:;+*%?!&$\(\)_#=~\'"@]*(?=[\s>/<])))?'
    r')*\s*/?\s*(?=[<>])')

if __name__ == '__main__':
    foo = '<table cellspacing="0" cellpadding="0" style="border-collapse'
    starttag.match(foo)

### File: mutation_inside_cyclegc.py

In [ ]:
# The cycle GC collector can be executed when any GC-tracked object is
# allocated, e.g. during a call to PyList_New(), PyDict_New(), ...
# Moreover, it can invoke arbitrary Python code via a weakref callback.
# This means that there are many places in the source where an arbitrary
# mutation could unexpectedly occur.

# The example below shows list_slice() not expecting the call to
# PyList_New to mutate the input list.  (Of course there are many
# more examples like this one.)


import weakref

class A(object):
    pass

def callback(x):
    del lst[:]


keepalive = []

for i in range(100):
    lst = [str(i)]
    a = A()
    a.cycle = a
    keepalive.append(weakref.ref(a, callback))
    del a
    while lst:
        keepalive.append(lst[:])

### File: recursive_call.py

In [ ]:
#!/usr/bin/env python3

# No bug report AFAIK, mail on python-dev on 2006-01-10

# This is a "won't fix" case.  It is known that setting a high enough
# recursion limit crashes by overflowing the stack.  Unless this is
# redesigned somehow, it won't go away.

import sys

sys.setrecursionlimit(1 << 30)
f = lambda f:f(f)

if __name__ == '__main__':
    f(f)

### File: trace_at_recursion_limit.py

In [ ]:
"""
From http://bugs.python.org/issue6717

A misbehaving trace hook can trigger a segfault by exceeding the recursion
limit.
"""
import sys


def x():
    pass

def g(*args):
    if True: # change to True to crash interpreter
        try:
            x()
        except:
            pass
    return g

def f():
    print(sys.getrecursionlimit())
    f()

sys.settrace(g)

f()

### File: underlying_dict.py

In [ ]:
import gc

thingy = object()
class A(object):
    def f(self):
        return 1
    x = thingy

r = gc.get_referrers(thingy)
if "__module__" in r[0]:
    dct = r[0]
else:
    dct = r[1]

a = A()
for i in range(10):
    a.f()
dct["f"] = lambda self: 2

print(a.f()) # should print 1